# Notebook 05 — Coalescent Theory: Intuition

**Module:** 06 — Genetics and Evolution  
**Notebook:** 05 of 12  
**Prerequisites:** Module 05 NB07 (Wright-Fisher), NB04 this module (fixation)  
**Time estimate:** 75 minutes

> **Tier 2 depth note:** The goal is intuition and simulation, not formal proof.
> The coalescent appears in ancient DNA papers, PSMC demographic inference, and
> population structure analyses — you need to know what it is and what it estimates.

---
## Step 1 — Motivation

The Wright-Fisher model looks *forward* in time: alleles change frequency generation
by generation. The **coalescent** looks *backward* in time: given a sample of sequences,
when did they last share a common ancestor? This backwards view is remarkably powerful
— it lets you estimate effective population sizes from genetic diversity alone, and
it underlies tools like PSMC, MSMC, and ARGweaver that reconstruct demographic history
from modern genomes.

---
## Step 2 — Intuition

Imagine running the Wright-Fisher model backwards. In each generation backwards,
two lineages that happen to share the same parent 'coalesce' — they merge into one
ancestral lineage. Starting from n sampled sequences, we trace backwards until all
lineages have merged into a single **most recent common ancestor (MRCA)**.

The key insight: with n lineages, each pair can coalesce with probability 1/(2Nₑ)
per generation. There are C(n,2) = n(n-1)/2 pairs, so the total coalescence rate
is n(n-1)/(4Nₑ) per generation. Small populations → fast coalescence → less
diversity. Large populations → slow coalescence → more diversity.

---
## Step 3 — Biological Background

**Key coalescent results:**

| Quantity | Formula | Meaning |
|---------|---------|--------|
| E[T_MRCA] (n=2) | 2Nₑ generations | Expected time to MRCA for 2 sequences |
| E[T_MRCA] (n samples) | 4Nₑ(1 - 1/n) generations | Time to MRCA for n sequences |
| Nucleotide diversity θ | 4Nₑμ | Directly encodes Nₑ and mutation rate μ |
| Watterson's θ_W | S/a₁ | Estimated from number of segregating sites |

**Applications of the coalescent:**
- **PSMC / MSMC:** infer Nₑ(t) — how effective population size changed over time —
  from a single diploid genome or a pair of genomes
- **Ancient DNA:** estimate when a population diverged from modern ones
- **Population structure:** detect admixture events and migration
- **ARG (Ancestral Recombination Graph):** extends coalescent to include recombination

**Human demographic history from the coalescent:**
- Human Nₑ ≈ 10,000 (much smaller than census size of 8 billion)
- African populations: larger Nₑ, older MRCA, more diversity
- Non-African populations: bottleneck during Out-of-Africa (≈50,000 years ago)
  → smaller Nₑ, younger MRCA, less diversity

---
## Step 4 — Mathematical Explanation

**Kingman's coalescent (1982):**

With n lineages in a population of size Nₑ, the time until the first coalescence event
is exponentially distributed:

$$T_n \sim \text{Exponential}\left(\binom{n}{2} \frac{1}{N_e}\right)$$

The expected waiting time to go from k to k-1 lineages:
$$E[T_k] = \frac{4N_e}{k(k-1)}$$

Total expected time to MRCA:
$$E[T_{MRCA}] = \sum_{k=2}^{n} \frac{4N_e}{k(k-1)} = 4N_e \left(1 - \frac{1}{n}\right)$$

---
## Step 6 — Python Implementation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import comb

In [ ]:
# Cell 6.1 — Kingman's coalescent simulation
def simulate_coalescent(n: int, Ne: int, seed: int = 42) -> list:
    """
    Simulate a genealogy under Kingman's coalescent.

    Returns a list of (time_of_event, lineages_before, lineages_after) tuples.
    Time is measured in generations going backward.
    """
    rng = np.random.default_rng(seed)
    lineages = n
    t = 0  # total time elapsed (backwards)
    events = []

    while lineages > 1:
        # Rate of coalescence: C(k,2) / Ne
        n_pairs = comb(lineages, 2)
        rate = n_pairs / Ne
        # Waiting time: exponential with the coalescence rate
        waiting_time = rng.exponential(scale=1.0 / rate)
        t += waiting_time
        events.append({
            'time': t,
            'lineages_before': lineages,
            'lineages_after': lineages - 1,
            'waiting_time': waiting_time,
        })
        lineages -= 1

    return events

Ne = 10_000
n = 10
events = simulate_coalescent(n=n, Ne=Ne)

print(f"Coalescent genealogy: n={n} samples, Ne={Ne:,}")
print(f"{'Lineages':>10} {'Waiting time (gen)':>20} {'Cumulative time':>16}")
print("-" * 48)
for ev in events:
    print(f"  {ev['lineages_before']:>5} → {ev['lineages_after']:<5} "
          f"{ev['waiting_time']:>16,.0f}     {ev['time']:>12,.0f}")
print(f"\nT_MRCA = {events[-1]['time']:,.0f} generations")
expected_tmrca = 4 * Ne * (1 - 1/n)
print(f"Expected T_MRCA = 4Ne(1-1/n) = {expected_tmrca:,.0f} generations")

In [ ]:
# Cell 6.2 — Distribution of T_MRCA across many realisations
n_realisations = 5000
tmrca_values = []
for seed in range(n_realisations):
    evs = simulate_coalescent(n=5, Ne=Ne, seed=seed)
    tmrca_values.append(evs[-1]['time'])

tmrca_values = np.array(tmrca_values)
expected_n5 = 4 * Ne * (1 - 1/5)
print(f"T_MRCA distribution (n=5, Ne={Ne:,}, {n_realisations} realisations):")
print(f"  Mean: {tmrca_values.mean():,.0f}  (expected: {expected_n5:,.0f})")
print(f"  Median: {np.median(tmrca_values):,.0f}")
print(f"  SD: {tmrca_values.std():,.0f}")
print(f"  Note: high variance — the coalescent is highly stochastic")

In [ ]:
# Cell 6.3 — Nucleotide diversity θ = 4Ne * mu
mu = 1.2e-8  # per base per generation (human germline)

print("Nucleotide diversity θ = 4Ne·μ for different Ne:")
print(f"  {'Ne':>12} {'θ (per site)':>15} {'Expected pairwise diff per kb':>32}")
print("-" * 62)
for Ne_vals in [1_000, 5_000, 10_000, 100_000, 1_000_000]:
    theta = 4 * Ne_vals * mu
    diffs_per_kb = theta * 1000
    print(f"  {Ne_vals:>12,} {theta:>15.5f} {diffs_per_kb:>32.3f}")
print()
print(f"Human Nₑ ≈ 10,000 → θ ≈ {4*10000*mu:.5f} → ~{4*10000*mu*1000:.2f} differences per kb")
print("(Observed human diversity ≈ 1 difference per 1000 bp → consistent!)")

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — T_MRCA distribution + expected waiting times per coalescence event
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Panel 1: T_MRCA distribution
ax = axes[0]
ax.hist(tmrca_values / 1000, bins=60, color='steelblue', alpha=0.8, density=True)
ax.axvline(expected_n5 / 1000, color='tomato', lw=2, ls='--',
           label=f'Expected T_MRCA = {expected_n5/1000:.0f}k gen')
ax.axvline(np.median(tmrca_values) / 1000, color='seagreen', lw=2, ls=':',
           label=f'Median = {np.median(tmrca_values)/1000:.0f}k gen')
ax.set_xlabel('T_MRCA (thousands of generations)')
ax.set_ylabel('Density')
ax.set_title(f'Distribution of T_MRCA (n=5, Ne={Ne:,})\nHigh variance: no single "expected" tree')
ax.legend(fontsize=8)

# Panel 2: Expected waiting time per coalescence step vs. n
ax = axes[1]
n_range = np.arange(2, 16)
expected_waits = [4 * Ne / (k * (k-1)) for k in n_range]
ax.bar(n_range, np.array(expected_waits) / 1000, color='steelblue', alpha=0.8)
ax.set_xlabel('Number of lineages k')
ax.set_ylabel('Expected waiting time (thousands of generations)')
ax.set_title('Coalescent waiting times:\nMost evolution happens in the last few lineages')
ax.annotate('← Most time\nspent here', (2, expected_waits[0]/1000 * 0.8),
            xytext=(4, expected_waits[0]/1000 * 0.6),
            arrowprops=dict(arrowstyle='->', color='tomato'), color='tomato')

plt.tight_layout()
plt.show()

---
## Step 8 — Exercises

1. Implement `simulate_coalescent(n, Ne)` from scratch. For n=2 and Ne=10,000,
   verify that the mean of 10,000 simulations ≈ 2Ne = 20,000 generations.
2. The effective population size of Neanderthals is estimated at ~3,000–5,000.
   Using θ = 4Nₑμ with μ=1.2×10⁻⁸, what level of nucleotide diversity would
   you expect in Neanderthal DNA? Compare to modern humans.
3. PSMC infers Nₑ(t) from a single diploid genome. What aspect of the coalescent
   does it use, and why can it detect demographic events that occurred thousands
   of generations ago?
4. An Out-of-Africa bottleneck reduced the ancestral population to N_bottle=500
   for 200 generations. Simulate a coalescent with these demographics (change Nₑ
   mid-simulation). How does the genealogy shape change?

---
## Quiz — Active Recall

1. What does the coalescent look at that the Wright-Fisher model does not?
2. What is the MRCA? How does Nₑ affect the expected time to MRCA?
3. What is nucleotide diversity θ? What two quantities determine it?
4. Why is Nₑ ≈ 10,000 for humans, not 8 billion?
5. Name one tool that uses the coalescent to infer demographic history from sequence data.

---
## Papers Referenced

- Kingman (1982). DOI: 10.1016/0304-4149(82)90011-4

---
## Reflection

**Date completed:** ____________________

> *[A paper uses PSMC to show that a population went through a bottleneck 50,000
> years ago. What data did they use? What did the coalescent tell them about it?]*

---
*Next: `06_phylogenetic_trees_distance_methods.ipynb`*